# Ejemplo de red bayesiana

Simulacion para determinar si  un estudiante apueba o desaprueba un curso

Horas de estudio
        →
    Asistencia
        →
Nota en el examen
        →
    Aprobacion


In [2]:
!pip install pgmpy


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Librerias de redes bayesiana
# Librería para crear la red bayesiana
from pgmpy.models import DiscreteBayesianNetwork
# Librería para crear las tablas de probabilidades condicionales 
from pgmpy.factors.discrete import TabularCPD
# Librería para realizar inferencia en la red bayesiana
from pgmpy.inference import VariableElimination

In [4]:
# Crear la estructura de la red bayesiana
# HorasEstudio influye en Asistencia
# Asistencia influye en NotaExamen
# NotaExamen influye en Aprobacion
model = DiscreteBayesianNetwork([
    ("HorasEstudio", "Asistencia"),
    ("Asistencia", "NotaExamen"),
    ("NotaExamen", "Aprobacion")
])

In [5]:
# Crear la tabla de probabilidades para la variable HorasEstudio
# Se considera que existe:
# 60% de probabilidad de que el estudiante estudie pocas horas
# 40% de probabilidad de que el estudiante estudie muchas horas
cpd_horas = TabularCPD(
    variable="HorasEstudio",
    variable_card=2,
    values=[[0.6],[0.4]],
    state_names={"HorasEstudio": ["Pocas", "Muchas"]}
)

In [6]:
# Crear la tabla de probabilidades condicionales para la variable Asistencia
# La asistencia depende de las horas de estudio.
# Si estudia pocas horas:
# 80% de probabilidad de asistencia baja
# 20% de probabilidad de asistencia alta
# Si estudia muchas horas:
# 30% de probabilidad de asistencia baja
# 70% de probabilidad de asistencia alta

cpd_asistencia = TabularCPD(
    variable="Asistencia",
    variable_card=2,
    values=[[0.8, 0.3],[0.2, 0.7]],
    evidence=["HorasEstudio"],
    evidence_card=[2],
    state_names={"Asistencia": ["Baja", "Alta"],"HorasEstudio": ["Pocas", "Muchas"]}
)

In [7]:
# Crear la tabla de probabilidades condicionales para la variable NotaExamen
# La nota del examen depende de la asistencia del estudiante.
# Si la asistencia es baja:
# 70% de probabilidad de obtener nota baja
# 30% de probabilidad de obtener nota alta
# Si la asistencia es alta:
# 20% de probabilidad de obtener nota baja
# 80% de probabilidad de obtener nota alta

cpd_nota = TabularCPD(
    variable="NotaExamen",
    variable_card=2,
    values=[
        [0.7, 0.2],[0.3, 0.8]],
    evidence=["Asistencia"],
    evidence_card=[2],
    state_names={"NotaExamen": ["Baja", "Alta"],"Asistencia": ["Baja", "Alta"]}
)

In [8]:
# Crear la tabla de probabilidades condicionales para la variable Aprobacion
# La aprobación depende de la nota obtenida en el examen.
# Si la nota es baja:
# 95% de probabilidad de desaprobar
# 5% de probabilidad de aprobar
# Si la nota es alta:
# 10% de probabilidad de desaprobar
# 90% de probabilidad de aprobar

cpd_aprobacion = TabularCPD(
    variable="Aprobacion",
    variable_card=2,
    values=[[0.95, 0.1],[0.05, 0.9]],
    evidence=["NotaExamen"],
    evidence_card=[2],
    state_names={"Aprobacion": ["Desaprueba", "Aprueba"],"NotaExamen": ["Baja", "Alta"]}
)

In [9]:
# Agregar al modelo todas las tablas de probabilidades condicionales
model.add_cpds(cpd_horas, cpd_asistencia, cpd_nota, cpd_aprobacion)
# Verificar que la red bayesiana y sus probabilidades sean correctas
model.check_model()

True

In [10]:
# Mostrar un mensaje indicando que el modelo fue creado correctamente
print("Modelo de red bayesiana creado y verificado correctamente.")
# Mostrar las relaciones existentes entre las variables
print(model.edges())

Modelo de red bayesiana creado y verificado correctamente.
[('HorasEstudio', 'Asistencia'), ('Asistencia', 'NotaExamen'), ('NotaExamen', 'Aprobacion')]


In [11]:
# Inferencia en la red bayesiana
# Se crea el motor de inferencia para poder realizar consultas al modelo
inferencia = VariableElimination(model)
# Caso del estudiante:
# Estudia muchas horas
# Tiene asistencia alta
# Tiene una nota alta en el examen
# Se desea conocer la probabilidad de que apruebe o desapruebe
# teniendo en cuenta estas evidencias.

resultado_aprobacion = inferencia.query(
    variables=["Aprobacion"],
    evidence={"HorasEstudio": "Muchas","Asistencia": "Alta","NotaExamen": "Alta"}
)

print("Probabilidad de aprobación del estudiante con evidencia:")
print(resultado_aprobacion)

Probabilidad de aprobación del estudiante con evidencia:
+------------------------+-------------------+
| Aprobacion             |   phi(Aprobacion) |
+========================+===================+
| Aprobacion(Desaprueba) |            0.1000 |
+------------------------+-------------------+
| Aprobacion(Aprueba)    |            0.9000 |
+------------------------+-------------------+


In [14]:
# Inferencia para un estudiante con alta probabilidad de desaprobar
# Evidencias:
# Estudia pocas horas
# Tiene asistencia baja
# Tiene una nota baja en el examen

resultado_desaprobacion = inferencia.query(
    variables=["Aprobacion"],
    evidence={"HorasEstudio": "Pocas","Asistencia": "Baja","NotaExamen": "Baja"    }
)

print("Probabilidad de desaprobación del estudiante:")
print(resultado_desaprobacion)

Probabilidad de desaprobación del estudiante:
+------------------------+-------------------+
| Aprobacion             |   phi(Aprobacion) |
+========================+===================+
| Aprobacion(Desaprueba) |            0.9500 |
+------------------------+-------------------+
| Aprobacion(Aprueba)    |            0.0500 |
+------------------------+-------------------+
